In [1]:
from grad_cam import *
import torchvision.models as models
import torch.nn as nn
import matplotlib.cm as cm
import torch
import cv2
from torchvision import datasets, models, transforms
from efficientnet_pytorch import EfficientNet
import os.path as osp
from PIL import Image
torch.cuda.set_device(1)
import pandas as pd

In [2]:
def get_device(cuda):
    cuda = cuda and torch.cuda.is_available()
    device = torch.device("cuda" if cuda else "cpu")
    if cuda:
        current_device = torch.cuda.current_device()
        print("Device:", torch.cuda.get_device_name(current_device))
    else:
        print("Device: CPU")
    return device
def preprocess(image_path):
    raw_image = cv2.imread(image_path)
    raw_image = cv2.resize(raw_image, (480,) * 2)
    image = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )(raw_image[..., ::-1].copy())
#     print(image.size())
#     print(raw_image.size)
    return image, raw_image
def get_classtable():
    classes = []
    with open("/mnt/HDD3/Users/dasleo/data/unsplit_long_bone/eff_maf_wrong/class_labels.txt") as lines:
        for line in lines:
            line = line.strip().split(" ", 1)[1]
            line = line.split(", ", 1)[0].replace(" ", "_")
            classes.append(line)
    return classes
def save_gradcam(filename, gcam, raw_image, paper_cmap=False):
    gcam = gcam.cpu().numpy()
    cmap = cm.jet_r(gcam)[..., :3] * 255.0
    if paper_cmap:
        alpha = gcam[..., None]
        gcam = alpha * cmap + (1 - alpha) * raw_image
    else:
        gcam = (cmap.astype(np.float) + raw_image.astype(np.float)) / 2
    cv2.imwrite(filename, np.uint8(gcam))
def initialize_model(structure,
                     num_classes,
                     use_pretrained=True):
    model_ft = None
    if structure == "efficientnet-b5":
        model_ft = EfficientNet.from_pretrained('efficientnet-b5',num_classes=num_classes)
        model_ft._fc = nn.Linear(in_features=2048,
                                out_features=2,
                                bias=True)
    if structure == "resnet50":
        model_ft = models.resnet50(pretrained=use_pretrained)
        model_ft.fc = nn.Linear(in_features=2048,
                                out_features=2,
                                bias=True)
    if structure == "densenet201":
        model_ft = models.densenet201(pretrained=use_pretrained)
        model_ft.classifier = nn.Linear(in_features=1920,
                                        out_features=2,
                                        bias=True)
    return model_ft

In [3]:
classes = get_classtable()

In [4]:
LIST=pd.read_csv('/mnt/HDD3/Users/dasleo/ipynbs/Model Testing/res_wrong_des_and_maf_right.csv')
# LIST=LIST.sample(n=30,random_state=0)
LIST.shape

(9, 4)

In [5]:
LIST.head()

,File_Name,CV,Label,Source_PATH
0,164_train.png,4,positive,/mnt/HDD3/Users/dasleo/data/CV_NO_Val/CV_4/tes...
1,208_val.png,5,positive,/mnt/HDD3/Users/dasleo/data/CV_NO_Val/CV_5/tes...
2,107314189876569_train.png,6,positive,/mnt/HDD3/Users/dasleo/data/CV_NO_Val/CV_6/tes...
3,355844569660292_train.png,6,positive,/mnt/HDD3/Users/dasleo/data/CV_NO_Val/CV_6/tes...
4,68_train.png,7,positive,/mnt/HDD3/Users/dasleo/data/CV_NO_Val/CV_7/tes...


In [6]:
k=0
for i,r in LIST[3*k:3*k+3].iterrows():
    print(torch.cuda.memory_allocated()/((1000**2)))
    CV=r['CV']
    CV=str(CV)
    state_dict_path='/mnt/HDD3/Users/dasleo/models/round3/Cross_Validation/resnet50/MURA_Anim_Finetune/'
    state_dict_path=state_dict_path+'CV_'+CV+"_best.h5"
    model=initialize_model(structure='resnet50',num_classes=2)
    model.to("cuda")
    model.eval();
    model.load_state_dict(torch.load(state_dict_path))
    image_path=r['Source_PATH']
    images = []
    raw_images = []
    print("Images:")
    print("{}".format(image_path))
    image, raw_image = preprocess(image_path)
    images.append(image)
    raw_images.append(raw_image)
    images = torch.stack(images).to("cuda")
    target_layers = ["layer4.1"]
#     if r['Label']=='negative':
#         target_class = 0 #negative
#     if r['Label']=='positive':
    target_class = 1 #positive
    output_dir='/mnt/HDD3/Users/dasleo/data/unsplit_long_bone/DF_MAF_COMP/res wrong des eff right/'
    F_name=r['File_Name']
    gcam = GradCAM(model=model)
    probs, ids = gcam.forward(images)
    print(probs,ids)
    ids_ = torch.LongTensor([[target_class]] * len(images)).to("cuda")
    gcam.backward(ids=ids_)
    for target_layer in target_layers:
        print("Generating Grad-CAM @{}".format(target_layer))
        # Grad-CAM
        print(target_layer)
        regions = gcam.generate(target_layer=target_layer)
        for j in range(len(images)):
            print(
                "\t#{}: {} ({:.5f})".format(
                    j, classes[target_class], float(probs[ids == target_class])
                )
            )

            save_gradcam(
                filename=osp.join(
                    output_dir,
                    "{}-{}-gradcam-{}-{}.png".format(
                        target_layer, "res", F_name, classes[target_class]
                    ),
                ),
                gcam=regions[j, 0],
                raw_image=raw_images[j],
        )
    del model
    del images
    del ids_
    del gcam
    del probs
    del ids
    del regions
    torch.cuda.empty_cache()


0.0
Images:
/mnt/HDD3/Users/dasleo/data/CV_NO_Val/CV_4/test/positive/164_train.png
tensor([[0.9011, 0.0989]], device='cuda:1', grad_fn=<SortBackward>) tensor([[0, 1]], device='cuda:1')
Generating Grad-CAM @layer4.1
layer4.1
	#0: positive (0.09893)
1161.78944
Images:
/mnt/HDD3/Users/dasleo/data/CV_NO_Val/CV_5/test/positive/208_val.png
tensor([[0.9595, 0.0405]], device='cuda:1', grad_fn=<SortBackward>) tensor([[1, 0]], device='cuda:1')
Generating Grad-CAM @layer4.1
layer4.1
	#0: positive (0.95951)
2325.938176
Images:
/mnt/HDD3/Users/dasleo/data/CV_NO_Val/CV_6/test/positive/107314189876569_train.png
tensor([[0.8747, 0.1253]], device='cuda:1', grad_fn=<SortBackward>) tensor([[1, 0]], device='cuda:1')
Generating Grad-CAM @layer4.1
layer4.1
	#0: positive (0.87474)
